# Tutorial IV: Introduction to LLM API Calls
<p>
Bern Winter School on Machine Learning, 2025<br>
Prepared by Mykhailo Vladymyrov and Matthew Vowels.
</p>

This work is licensed under a <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License</a>.

In this tutorial session we will get familiar wtih:
* How to make calls to OpenAI's GPT API
* How to make use of local LLMs with langchain and Ollama


Background:

1. What is an API?
An API (Application Programming Interface) allows different software to communicate, enabling access to services like LLMs without managing infrastructure. APIs simplify usage, scale easily, and offer accessibility, but sending data to external servers raises privacy concerns. Usually to use an API, you will need an API key.

2. Why Run Local Models?
Running LLMs locally is useful when dealing with sensitive data, high API costs, or needing offline access and customization. However, it requires significant hardware, technical setup, and may lack scalability and performance compared to cloud-based APIs.

3. Model Providers vs. Frameworks
Model Providers like OpenAI https://platform.openai.com/docs/overview ,  Hugging Face https://huggingface.co/ , and Ollama https://ollama.com/ develop and host LLMs, either via APIs or tools for local use. Frameworks like LangChain https://python.langchain.com/ focus on integrating LLMs into workflows, offering tools for prompt engineering, memory, and task chaining. Providers supply models, while frameworks enhance usability.

Today we will look at local models with the Ollama model provider, as well as the OpenAI API for a remote/3rd party provider.

We will use langchain, but note that other libraries exist (e.g. llamaindex or haystack).



### NOTE:
Even the smallest local models will take some time to run.

### References:
https://medium.com/@abonia/running-ollama-in-google-colab-free-tier-545609258453

https://python.langchain.com/docs/integrations/llms/openai/

https://youtu.be/7xTGNNLPyMI?si=PczttqFRiDj0LcNM

https://arxiv.org/html/2408.05093v1



# Some Background

The LLMs we interact with regularly which are served by e.g. OpenAI are significantly more complex than the 'base models' on which they are built.

To summarise, a production-ready LLM system has been through the following stages:

**Pre-Training**:

Here a 'base model' is trained in next-token prediction, using a large set of diverse corpora from the internet. The model cannot provide 'chat-style' responses.

https://app.hyperbolic.xyz/ to play with some base models (requires signup). Alternatively, see https://www.together.ai/ and https://aistudio.google.com/prompts/new_chat

And see what happens if you kickstart a base-model with the beginning of a wiki page...

**Supervised Fine Tuning (SVT)**

Here the model is fine-tuned for specific tasks like Q&A, summarization, coding.

**Further Post-Training, RL, and Alignment**

Post-training refers to additional steps that come after the initial pre-training of a large language model to further adapt, align, or improve it for specific tasks, domains, or user preferences (e.g. reinforcement learning like RHLF, domain adaptation).


Here the prompts can be modified to ensure certain behaviour / information is accessible to the model. Can include *distillation*, whereby a student model learns from a teacher model’s output probabilities or labels.
Example: The teacher model outputs probabilities across multiple classes (e.g., 80% cat, 15% dog, 5% other), and the student learns to match this distribution. Helps develop small models from larger models, hence the term 'distillation'.



As such, the final models you interact with have likely been through many steps of refinement - they are not 'simply' trained on the internet.


### What does the LLM look like?

https://bbycroft.net/llm



## 0. Install Some Base Dependencies

Most importantly, we will need langchain-ollama dna lanchain-openai.

In [ ]:
# install the langchain framework for local ollama and remote openai calls:
%pip install -U langchain-ollama langchain-openai

# note that langchain also provide interfaces for other providers/APIs like langchain-anthropic and langchain-huggingface

In [ ]:
# install colab-xterm to be able to interact with the terminal in the notebook (it might be useful)
!pip install colab-xterm
%load_ext colabxterm

Now we need to install Ollama within the terminal. Execute the next cell, and then once in the terminal, run this:

```
sudo apt-get install zstd
curl -fsSL https://ollama.com/install.sh | sh
```

And, once this is completed, start the Ollama server with:

```ollama serve &```

The `&` tells it to run in the background.

In [ ]:

%xterm

# 1. Download a local model with Ollama

This is very easy, and Ollama provides a huge selection to choose from:

For example we will start with a small model 'Qwen':
https://ollama.com/library/qwen2.5:0.5b

In [ ]:
!ollama pull qwen2.5:0.5b

In [ ]:
# you can see the downloaded models here:

!ollama list

You can check out some of the other available models e.g. here:

https://ollama.com/library/llama3.1/tags


Now back to Python.....

## 2. Run a simple prompt + query

Whenever you interact with ChatGPT, your messages are injected into a prompt. The prompt is, itself, just another set of text instructions, which help to determine the nature of the output of the model.


Langchain makes it easy to construct/modify prompts. This can be particularly useful if we want to do dynamic prompting, where the output of the model can be used to influence the contents of the subsequent prompt.


Chatbots do this, because their prompts ought to contain the message history (or some summary thereof) for them to be able to maintain knowledge of the context of the conversation when you send them the next message. Otherwise, you yourself would have to copy and prepend the chat history with your next message (LLMs do not, by default, have memory, although this is open area of research).



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

local_model = OllamaLLM(model="qwen2.5:0.5b")  # langchain provides a way to interface with our Ollama model provider (as they also do with OpenAI and Hugging Face)


See below - the prompt formatting includes placeholders with speific variable names (in the exampe below, 'question' is in {} and can be set dynamically).

Note that in this prompt template you can include all sorts of information about how the model should behave etc.

In [ ]:

template = "You are a helpful assistant. Answer the following question: {question}"


We can give this template to the langchain 'from_template' method:

In [ ]:
prompt_template = ChatPromptTemplate.from_template(template)

We can then create a 'chain' with langCHAIN.  This is a series of operations.

We keep it simple below, but more examples can be found here:

https://github.com/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/02-langchain-chains.ipynb

In [ ]:
local_chain = prompt_template | local_model


In this chain, we simple say that whatever we pass to it will first go through the prompt, and then to the model.

We can 'invoke' the chain, which allows us to pass information to it. The next cell will take a little time.

In [ ]:

local_chain.invoke({"question": "What is an LLM?"})

In the example above, note that 'question' is the name of the variable we defined in the prompt, and we have the text which should be injected into the associated part of the prompt.


Here is a more complex example. You can see how you might create your own templates and inject information into the associated prompt.


In [ ]:
template = """
You are a helpful assistant. Answer the following:
Question: {question}
The topic is: {topic}
Please format the text as follows: {format}
"""
prompt_template = ChatPromptTemplate.from_template(template)
local_chain = prompt_template | local_model

response = local_chain.invoke({
    "question": "What is a capital letter?",
    "topic": "TESTING CAPITAL LETTERS",
    "format": "USE CAPITAL LETTERS IN THE RESPONSE"
})


In [ ]:
response

Note that all the tokenization is handled automatically.

For a reminder about tokenization, see: https://tiktokenizer.vercel.app/?model=cl100k_base


In this case, you might find that small models aren't very compliant...

## 3. OpenAI / 3rd Party Example


Firstly, beware of privacy laws before sharing people's potentially sensitive data!

Secondly, for this you will need to create an openAI API key.

https://platform.openai.com/api-keys

And add some credits to your account.....


Make sure you keep your API keys private!

DO NOT add them to code which will later be pushed to a github repository! Keep them local (e.g. using an environment variable).

When you aren't using google colab, you can do something like this:

### Windows

Open Command Prompt (or PowerShell).

`setx OPENAI_API_KEY "your-api-key-here"`

Restart your terminal or system for the change to take effect.


### Linux / Mac

Open a terminal.

Add the variable to your shell configuration file (`~/.bashrc`, `~/.zshrc`, or `~/.bash_profile`):

```
export OPENAI_API_KEY="your-api-key-here"
```

Reload the config:

```
source ~/.bashrc  # For bash
source ~/.zshrc   # For zsh
```



## In Python

Once back in python, the key can be used like this:


```
import os
api_key = os.getenv("OPENAI_API_KEY")
```


To use colab secret - enable acces form the notebook, then do:

In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI
openai_model = ChatOpenAI(model='gpt-3.5-turbo', api_key=api_key)

# a list of models can be found here: https://platform.openai.com/docs/models  but note they will have different costs associated with them

template = "You are a helpful assistant. Answer the following question: {question}"
prompt_template = ChatPromptTemplate.from_template(template)

openai_chain = prompt_template | openai_model
response = openai_chain.invoke({"question": "Givne that you are GPT 3.5 turbo, how much do you cost per 1000 tokens?"})
print(response.response_metadata)
print(response.content)

In [ ]:
template = """
You are a helpful assistant. Answer the following:
Question: {question}
The topic is: {topic}
Please format the text as follows: {format}
"""
prompt_template = ChatPromptTemplate.from_template(template)
openai_chain = prompt_template | openai_model

response = openai_chain.invoke({
    "question": "What is a capital letter?",
    "topic": "TESTING CAPITAL LETTERS",
    "format": "USE CAPITAL LETTERS IN THE RESPONSE"
})
response.content

In [ ]:
# as of feb 2024:  ‘gpt-3.5-turbo-0125’ costs $0.0005/1k input and $0.0015/1k out
print(response.usage_metadata['input_tokens'])
print(response.usage_metadata['output_tokens'])

### Quick Exercise

Write code which has 1 template and 1 question, but asks for the response in a different tone (e.g. like a pirate).

In [ ]:
# your solution....


# 4. Multi-step Dynamic Prompting



In [ ]:

openai_model = ChatOpenAI(model="gpt-3.5-turbo", api_key=api_key)  # Use gpt-4 if available

summary_template = """
You are a helpful assistant. When does the Bern winter school start:
{text}
"""
summary_prompt_template = ChatPromptTemplate.from_template(summary_template)


translation_template = """
You are a helpful assistant, rewrite this in french:
{text}
"""
translation_prompt_template = ChatPromptTemplate.from_template(translation_template)

# Improved lambda function for passing output correctly
def extract_summary(output):
    summary_text = output.content  # Ensure it’s clean and properly formatted
    print(summary_text)
    return translation_prompt_template.format_prompt(text=summary_text)

# Create chain-like flow using | operator
openai_chain = (
    summary_prompt_template
    | openai_model  # First LLM
    | extract_summary  # Pass formatted output to the next prompt
    | openai_model  # Second LLM
)

# Test input
input_text = """
The Bern Winter School on Deep Learning is an annual educational program organized by the Data Science Lab at the University of Bern. Designed for researchers, students, and professionals from both the public and private sectors, the school aims to deepen participants' knowledge in machine learning and artificial intelligence. The upcoming session is scheduled from February 24 to February 28, 2025.
"""

# Run the chain
output = openai_chain.invoke({"text": input_text})
print(output.content)


# 5. What can we do with all this?

**a)** Create chatbots:

https://therapy-chatbot.unil.ch/c1/

**b)** Automatically annotate documents (some of these may be private but we can discuss them during the tutorial):


#### TheraCode: https://github.com/psifx/theracode/tree/main/theracode/tasc

This is a tool for taking psychotherapy transcripts as an input, and annotating them according to various linguistic cues in order to identify positive and negative types of therapist interventions.

#### psy-RAG:  https://github.com/psifx/psy-rag/
This is a tool for ingesting large databases of scientific articles and answering question about these documents (e.g. 'what is the sample size?').

#### Multimodal Audio/Video/Language Processing Pipelines: https://github.com/psifx/psifx


**c)** Parsing PDFs using e.g. GeminiFlash
https://medium.com/@woyera/how-i-use-gemini-on-my-pdf-files-using-python-50f4eaba4f0b

# Bonus w/ Gemini and PDF parsing

https://ai.google.dev/gemini-api/docs/migrate


The following code requires (as above) google aistudio api key.

It provides an example with a pdf document.


In [ ]:
import os
from google.colab import userdata
from google import genai
import requests
import pathlib


env_path = pathlib.Path().resolve().parent / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            key, value = line.strip().split('=', 1)
            os.environ[key] = value

# Access the environment variable
#api_key = os.getenv("GEMINI_API_KEY")

# on colab:
api_key = userdata.get('GEMINI_API_KEY')

# Configure GenAI with the API key
client = genai.Client(api_key=api_key)


def get_response(url, client, model_name, prompt):
    """Extract structured data from a scientific article PDF and return it as an Article object."""
    response = requests.get(url)

    # Save the PDF file locally with the original filename
    filename = url.split('/')[-1]  # Extract '2410.14485.pdf' from the URL
    with open(filename, 'wb') as file:
        file.write(response.content)

    # Upload the PDF file using the client library
    my_file = client.files.upload(file=filename)
    print(prompt)
    response = client.models.generate_content(
        model=model_name,
        contents=[prompt, my_file]
    )
    return response.text


pdf_path = 'https://arxiv.org/pdf/2410.14485.pdf'

model_name = 'gemini-2.0-flash-001'

prompt = 'Summarise the document.'

article = get_response(url=pdf_path, client=client, model_name=model_name, prompt=prompt)

print("Processing complete.")
